# EMT Dataset Generation (FEM-based)
Generates synthetic EMT data using `pyeit` FEM solver.
This notebook is the interactive version of `emt/generate_data.py`.

**Prerequisites:** `pip install pyeit h5py`

Output files written to `EMT_DATA_DIR` (set in `config.py`):
- `emt_train.h5` / `emt_test.h5`
- `sensitivity_matrix_A.npy`
- `baseline_voltages_v0.npy`
- `element_centroids.npy`
- `electrode_coords.npy`

> **NOTE:** This is a WORK IN PROGRESS. Dataset creation details (mesh resolution,
> number of samples, noise levels) are still being refined.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from config import EMT, EMT_DATA_DIR

import numpy as np
import matplotlib.pyplot as plt

print("EMT_DATA_DIR:", EMT_DATA_DIR)
EMT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Config (mirrors config.py EMT dict — edit config.py to change)
N_TRAIN    = EMT["n_train"]
N_TEST     = EMT["n_test"]
N_COILS    = EMT["n_coils"]
MESH_H0    = EMT["mesh_h0"]
DIST_EXC   = EMT["dist_exc"]
IMG_SIZE   = EMT["img_size"]
DOMAIN_R   = EMT["domain_r"]
SIGMA_BG   = EMT["sigma_bg"]
SIGMA_OBJ  = EMT["sigma_obj"]
OBJ_R_MIN  = EMT["obj_r_min"]
OBJ_R_MAX  = EMT["obj_r_max"]
NOISE_DB   = EMT["noise_db"]

print(f"N_TRAIN={N_TRAIN}  N_TEST={N_TEST}  COILS={N_COILS}  IMG={IMG_SIZE}x{IMG_SIZE}  SNR={NOISE_DB}dB")


## 1. Build FEM Mesh and Sensitivity Matrix A

In [ ]:
import pyeit.mesh     as mesh_lib
import pyeit.eit.protocol as protocol_lib
import pyeit.eit.fem   as fem_lib

print("Creating FEM mesh ...")
eit_mesh   = mesh_lib.create(n_el=N_COILS, h0=MESH_H0)
n_nodes    = eit_mesh.node.shape[0]
n_elements = eit_mesh.element.shape[0]
print(f"  {n_nodes} nodes, {n_elements} triangular elements")

el_pos_idx = eit_mesh.el_pos
el_coords  = eit_mesh.node[el_pos_idx, :2]

print("Setting up measurement protocol ...")
eit_protocol = protocol_lib.create(N_COILS, dist_exc=DIST_EXC, step_meas=1, parser_meas="std")
N_MEAS = eit_protocol.n_exc * eit_protocol.n_meas
print(f"  Total measurements: {N_MEAS}")

print("Computing Jacobian (sensitivity matrix A) ...")
fwd_solver = fem_lib.EITForward(eit_mesh, eit_protocol)
J, v0 = fwd_solver.compute_jac(perm=SIGMA_BG * np.ones(n_elements), normalize=False)
print(f"  Jacobian shape: {J.shape}")

row_norms = np.linalg.norm(J, axis=1, keepdims=True) + 1e-12
A = J / row_norms

CENTROIDS = eit_mesh.node[eit_mesh.element].mean(axis=1)[:, :2]

import numpy as np
np.save(EMT_DATA_DIR / "sensitivity_matrix_A.npy", A)
np.save(EMT_DATA_DIR / "baseline_voltages_v0.npy", v0)
np.save(EMT_DATA_DIR / "element_centroids.npy",    CENTROIDS)
np.save(EMT_DATA_DIR / "electrode_coords.npy",     el_coords)
print("Physics matrices saved.")


## 2. Visualise FEM Mesh and Sensitivity Map

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Mesh
axes[0].triplot(eit_mesh.node[:,0], eit_mesh.node[:,1], eit_mesh.element, color="gray", lw=0.4)
axes[0].scatter(el_coords[:,0], el_coords[:,1], c="red", s=60, zorder=5)
for i, (x, y) in enumerate(el_coords):
    axes[0].annotate(str(i+1), (x, y), fontsize=8, ha="center")
axes[0].set_title("FEM Mesh (8 electrodes)", fontsize=12)
axes[0].set_aspect("equal"); axes[0].axis("off")

# Sensitivity map (sum of |A| columns)
sens = np.abs(A).sum(axis=0)
axes[1].scatter(CENTROIDS[:,0], CENTROIDS[:,1], c=sens, cmap="hot", s=4)
axes[1].set_title("Sensitivity Map |A| sum", fontsize=12)
axes[1].set_aspect("equal"); axes[1].axis("off")

# A matrix image
im = axes[2].imshow(np.abs(A[:50, :50]), cmap="viridis", aspect="auto")
axes[2].set_title("A matrix (first 50x50 entries)", fontsize=12)
plt.colorbar(im, ax=axes[2])

plt.suptitle("FEM Mesh and Sensitivity Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("emt_fem_mesh.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Phantom Generator and Sample Visualisation

In [ ]:
xi = np.linspace(-1, 1, IMG_SIZE)
yi = np.linspace(-1, 1, IMG_SIZE)
XX, YY = np.meshgrid(xi, yi)
DOMAIN_MASK = (XX**2 + YY**2) <= DOMAIN_R**2

from scipy.interpolate import griddata

def make_phantom():
    n_objects  = np.random.randint(1, 4)
    perm_elem  = SIGMA_BG * np.ones(n_elements)
    placed     = []
    attempts   = 0
    while len(placed) < n_objects and attempts < 300:
        attempts += 1
        r = np.random.uniform(OBJ_R_MIN, OBJ_R_MAX)
        margin  = r + 0.04
        max_cen = DOMAIN_R - margin
        if max_cen <= 0: continue
        angle = np.random.uniform(0, 2*np.pi)
        dist  = np.random.uniform(0, max_cen)
        cx, cy = dist*np.cos(angle), dist*np.sin(angle)
        overlap = any(np.sqrt((cx-px)**2+(cy-py)**2) < (r+pr+0.03) for (px,py,pr) in placed)
        if overlap: continue
        in_circle = (CENTROIDS[:,0]-cx)**2 + (CENTROIDS[:,1]-cy)**2 <= r**2
        perm_elem[in_circle] = SIGMA_OBJ
        placed.append((cx, cy, r))
    delta = perm_elem - SIGMA_BG
    img = griddata(CENTROIDS, delta, (XX, YY), method="linear", fill_value=0.0)
    img[~DOMAIN_MASK] = 0.0
    return perm_elem, img.astype(np.float32), len(placed)

def add_noise(signal, snr_db):
    power = np.mean(signal**2)
    snr   = 10**(snr_db/10.0)
    std   = np.sqrt(power/(snr+1e-12))
    return signal + np.random.randn(*signal.shape).astype(np.float32)*std

# Sample visualisation
np.random.seed(42)
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for col in range(6):
    perm, img, n_obj = make_phantom()
    for row, n_v in enumerate([1, 2, 3]):
        pass  # placeholder — actual rows = 1-obj, 2-obj, 3-obj samples
    axes[0, col].imshow(img, cmap="hot")
    axes[0, col].set_title(f"n_obj={n_obj}", fontsize=9)
    axes[0, col].axis("off")

plt.suptitle("Sample EMT Phantoms (conductivity contrast)", fontsize=13)
plt.tight_layout()
plt.savefig("emt_sample_phantoms.png", dpi=120, bbox_inches="tight")
plt.show()


## 4. Generate Dataset

In [ ]:
# Run the full generation script (recommended) or generate here
# Option A (recommended): run from terminal
#     python emt/generate_data.py

# Option B: generate directly in this notebook
import sys
sys.path.insert(0, os.path.abspath('..'))
from generate_data import generate_split, setup
import argparse

class Args:
    n_train   = N_TRAIN
    n_test    = N_TEST
    n_coils   = N_COILS
    mesh_h0   = MESH_H0
    dist_exc  = DIST_EXC
    img_size  = IMG_SIZE
    domain_r  = DOMAIN_R
    sigma_bg  = SIGMA_BG
    sigma_obj = SIGMA_OBJ
    obj_r_min = OBJ_R_MIN
    obj_r_max = OBJ_R_MAX
    noise_db  = NOISE_DB

n_elem, n_meas = setup(Args())
train_imgs, train_meas, train_nobj = generate_split("train", Args.n_train, Args(), n_elem, n_meas)
test_imgs,  test_meas,  test_nobj  = generate_split("test",  Args.n_test,  Args(), n_elem, n_meas)
print("Dataset generation complete.")


## 5. Dataset Statistics

In [ ]:
import h5py

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, split in zip(axes[:2], ["train", "test"]):
    h5 = EMT_DATA_DIR / f"emt_{split}.h5"
    if h5.exists():
        with h5py.File(h5, "r") as f:
            n_obj = f["n_objects"][:]
        counts = [np.sum(n_obj == k) for k in [1, 2, 3]]
        ax.bar([1, 2, 3], counts, color="steelblue", alpha=0.8)
        ax.set_title(f"{split} — object count distribution", fontsize=12)
        ax.set_xlabel("# objects"); ax.set_ylabel("# samples")
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f"{split}.h5 not found", ha="center", va="center", transform=ax.transAxes)

# A matrix condition number
cond = np.linalg.cond(A)
axes[2].text(0.5, 0.5, f"A condition number:
{cond:.1e}
Shape: {A.shape}",
             ha="center", va="center", transform=axes[2].transAxes, fontsize=14)
axes[2].set_title("Sensitivity Matrix A Info", fontsize=12)

plt.suptitle("EMT Dataset Statistics", fontsize=14)
plt.tight_layout()
plt.savefig("emt_dataset_stats.png", dpi=120, bbox_inches="tight")
plt.show()
